# SmartGov AI — Complete Colab Prototype

This notebook creates and runs the SmartGov AI Flask website with five connected modules:
1. AI Citizen Assistant
2. Scheme Finder
3. Application Tracker
4. Grievance Management
5. Governance Dashboard

**Important:** This is a demo using fictional application records and a local SQLite database. It does not connect to live government databases or submit complaints to government offices.

Run cells from top to bottom.

In [ ]:
# 1. Install Flask
!pip -q install "Flask>=3.0,<4.0"
print("Flask installed.")

## 2. Create the project files

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/SmartGov_AI")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)

In [ ]:
from pathlib import Path
p = Path('requirements.txt')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('Flask>=3.0,<4.0\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('app.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('from flask import Flask, request, jsonify, render_template\nfrom services.storage import init_db\nfrom services.assistant import answer_question\nfrom services.schemes import search_schemes\nfrom services.tracker import get_application\nfrom services.grievances import create_grievance, list_grievances, update_grievance\nfrom services.dashboard import dashboard_data\n\napp = Flask(__name__)\n\n@app.get("/")\ndef home():\n    return render_template("index.html")\n\n@app.post("/api/assistant")\ndef assistant_api():\n    data = request.get_json(silent=True) or {}\n    question = (data.get("question") or "").strip()\n    language = (data.get("language") or "English").strip()\n    if not question:\n        return jsonify({"error": "Please enter a question."}), 400\n    return jsonify(answer_question(question, language))\n\n@app.post("/api/schemes")\ndef schemes_api():\n    data = request.get_json(silent=True) or {}\n    query = (data.get("query") or "").strip()\n    audience = (data.get("audience") or "citizen").strip().lower()\n    return jsonify({"results": search_schemes(query, audience)})\n\n@app.get("/api/applications/<application_id>")\ndef tracker_api(application_id):\n    record = get_application(application_id)\n    if not record:\n        return jsonify({"error": "Application not found. This prototype only contains demo records."}), 404\n    return jsonify(record)\n\n@app.post("/api/grievances")\ndef create_grievance_api():\n    data = request.get_json(silent=True) or {}\n    name = (data.get("citizen_name") or "Demo Citizen").strip()\n    description = (data.get("description") or "").strip()\n    locality = (data.get("locality") or "").strip()\n    if len(description) < 8:\n        return jsonify({"error": "Please describe the complaint in at least 8 characters."}), 400\n    return jsonify(create_grievance(name, description, locality)), 201\n\n@app.get("/api/grievances")\ndef list_grievances_api():\n    return jsonify({"grievances": list_grievances()})\n\n@app.patch("/api/grievances/<ticket_id>")\ndef update_grievance_api(ticket_id):\n    data = request.get_json(silent=True) or {}\n    status = (data.get("status") or "").strip()\n    result = update_grievance(ticket_id, status)\n    if not result:\n        return jsonify({"error": "Ticket not found or status is invalid."}), 404\n    return jsonify(result)\n\n@app.get("/api/dashboard")\ndef dashboard_api():\n    return jsonify(dashboard_data())\n\nif __name__ == "__main__":\n    init_db()\n    app.run(debug=True, use_reloader=False)\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('services/__init__.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('services/storage.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import sqlite3\nfrom pathlib import Path\n\nBASE_DIR = Path(__file__).resolve().parent.parent\nDATABASE_PATH = BASE_DIR / "smartgov.db"\n\ndef get_db():\n    connection = sqlite3.connect(str(DATABASE_PATH))\n    connection.row_factory = sqlite3.Row\n    connection.execute("PRAGMA foreign_keys = ON")\n    return connection\n\ndef init_db():\n    with get_db() as db:\n        db.execute("""\n            CREATE TABLE IF NOT EXISTS applications (\n                id INTEGER PRIMARY KEY AUTOINCREMENT,\n                application_id TEXT UNIQUE NOT NULL,\n                applicant_name TEXT NOT NULL,\n                service_name TEXT NOT NULL,\n                status TEXT NOT NULL,\n                last_updated TEXT NOT NULL,\n                next_step TEXT NOT NULL\n            )\n        """)\n        db.execute("""\n            CREATE TABLE IF NOT EXISTS schemes (\n                id INTEGER PRIMARY KEY AUTOINCREMENT,\n                name TEXT NOT NULL,\n                description TEXT NOT NULL,\n                audience TEXT NOT NULL,\n                keywords TEXT NOT NULL,\n                official_url TEXT NOT NULL,\n                eligibility_note TEXT NOT NULL\n            )\n        """)\n        db.execute("""\n            CREATE TABLE IF NOT EXISTS grievances (\n                id INTEGER PRIMARY KEY AUTOINCREMENT,\n                ticket_id TEXT UNIQUE NOT NULL,\n                citizen_name TEXT NOT NULL,\n                description TEXT NOT NULL,\n                category TEXT NOT NULL,\n                priority TEXT NOT NULL,\n                department TEXT NOT NULL,\n                locality TEXT DEFAULT \'\',\n                status TEXT NOT NULL,\n                possible_duplicate TEXT,\n                created_at TEXT NOT NULL,\n                updated_at TEXT NOT NULL\n            )\n        """)\n        applications = [\n            ("SG-1001", "Demo Citizen", "Income Certificate", "Under Review", "2026-10-10", "Check the relevant department for further updates."),\n            ("SG-1002", "Demo Citizen", "Residence Certificate", "Approved", "2026-10-09", "Follow the issuing authority instructions for collection."),\n            ("SG-1003", "Demo Citizen", "Student Assistance", "Pending Documents", "2026-10-08", "Review the document requirements before proceeding.")\n        ]\n        db.executemany("""\n            INSERT OR IGNORE INTO applications\n            (application_id, applicant_name, service_name, status, last_updated, next_step)\n            VALUES (?, ?, ?, ?, ?, ?)\n        """, applications)\n        schemes = [\n            ("National Scholarship Portal", "Explore scholarship opportunities and application resources.", "student", "scholarship education student financial assistance", "https://scholarships.gov.in/", "Eligibility depends on the specific scholarship and its official guidelines."),\n            ("PM-KISAN", "Access information about the PM-KISAN farmer income support scheme.", "farmer", "farmer agriculture income support pm kisan", "https://pmkisan.gov.in/", "Check the current official eligibility rules and exclusions."),\n            ("UMANG", "Discover digital public services through the UMANG platform.", "citizen", "citizen government services digital documents", "https://web.umang.gov.in/", "Service availability and eligibility vary by service."),\n            ("Pradhan Mantri Jan Dhan Yojana (PMJDY)", "Financial inclusion scheme providing access to basic banking and related services.", "citizen", "bank account financial inclusion pmjdy jan dhan money", "https://pmjdy.gov.in/scheme", "Confirm account and benefit conditions with the official portal or participating bank."),\n            ("Pradhan Mantri Jeevan Jyoti Bima Yojana (PMJJBY)", "Renewable life insurance scheme offered through participating banks and post offices.", "citizen", "life insurance pmjjby social security", "https://jansuraksha.gov.in/", "Review current age, premium, enrolment and claim rules on the official portal."),\n            ("Pradhan Mantri Suraksha Bima Yojana (PMSBY)", "Affordable accident insurance scheme available through participating institutions.", "citizen", "accident insurance pmsby social security", "https://jansuraksha.gov.in/", "Check current eligibility, premium and coverage terms before enrolment."),\n            ("Atal Pension Yojana (APY)", "Pension scheme for eligible subscribers to support income security in old age.", "citizen", "pension apy old age social security", "https://www.npscra.nsdl.co.in/scheme-details.php", "Eligibility and contribution amounts depend on current scheme rules."),\n            ("Pradhan Mantri Shram Yogi Maandhan (PM-SYM)", "Pension scheme information for eligible workers in the unorganised sector.", "worker", "pm sym pension worker unorganised trader social security", "https://maandhan.in/", "Check the official portal for current age, income and enrolment requirements."),\n            ("Pradhan Mantri Awas Yojana (PMAY)", "Official housing scheme information for eligible urban and rural households.", "citizen", "housing house home pmay awas urban rural", "https://pmay-urban.gov.in/", "Scheme components and application routes vary; check the relevant official portal."),\n            ("PM SVANidhi", "Working-capital support information for eligible street vendors.", "business", "street vendor loan working capital pmsvanidhi business", "https://pmsvanidhi.mohua.gov.in/", "Check the official portal for current lending and eligibility conditions."),\n            ("PM Vishwakarma", "Support for eligible traditional artisans and craftspeople, including skills and toolkit support.", "business", "artisan craftspeople skill toolkit vishwakarma traditional", "https://pmvishwakarma.gov.in/", "Eligible trades and benefits are defined by the official scheme guidelines."),\n            ("Pradhan Mantri Fasal Bima Yojana (PMFBY)", "Crop insurance scheme information for farmers.", "farmer", "crop insurance farming pmfby agriculture", "https://pmfby.gov.in/", "Crop, season, state and enrolment deadlines may vary; verify current notices."),\n            ("Pradhan Mantri MUDRA Yojana (PMMY)", "Information on loans for eligible micro and small non-farm enterprises through lending institutions.", "business", "mudra loan micro small business entrepreneur pmmy", "https://www.mudra.org.in/", "Loan approval and terms are decided by the lending institution."),\n            ("Startup India", "Government initiative offering resources and guidance for startup recognition and ecosystem support.", "business", "startup india entrepreneur new business funding recognition", "https://www.startupindia.gov.in/", "Benefits have separate eligibility criteria; review current official guidelines.")\n        ]\n        for item in schemes:\n            db.execute("""\n                INSERT INTO schemes (name, description, audience, keywords, official_url, eligibility_note)\n                SELECT ?, ?, ?, ?, ?, ?\n                WHERE NOT EXISTS (SELECT 1 FROM schemes WHERE name = ?)\n            """, (*item, item[0]))\n        db.commit()\n\nif __name__ == "__main__":\n    init_db()\n    print(f"Database initialized at {DATABASE_PATH}")\n', encoding='utf-8')
print('Updated', p)

In [ ]:
from pathlib import Path
p = Path('services/assistant.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('FAQS = [\n    {\n        "keywords": ["income certificate", "income proof"],\n        "answer": "For an income certificate, check your state government\'s official service portal or the relevant revenue office. Required documents and processing times vary by state.",\n        "url": "https://services.india.gov.in/"\n    },\n    {\n        "keywords": ["scholarship", "student aid"],\n        "answer": "You can explore eligible scholarship opportunities on the National Scholarship Portal. Read the current scheme guidelines before applying.",\n        "url": "https://scholarships.gov.in/"\n    },\n    {\n        "keywords": ["farmer", "pm kisan", "agriculture"],\n        "answer": "For farmer-related support and PM-KISAN information, use the official PM-KISAN portal and verify current eligibility requirements.",\n        "url": "https://pmkisan.gov.in/"\n    },\n    {\n        "keywords": ["complaint", "grievance", "complain"],\n        "answer": "You can submit a demo grievance through the SmartGov AI Grievance Management module. This prototype creates a local ticket and does not forward complaints to a government department.",\n        "url": "https://pgportal.gov.in/"\n    },\n    {\n        "keywords": ["application status", "track application", "tracking"],\n        "answer": "Use the Application Tracker with your reference number. This prototype only recognizes sample IDs such as SG-1001, SG-1002, and SG-1003.",\n        "url": "https://services.india.gov.in/"\n    },\n    {\n        "keywords": ["umang", "government services", "digital service"],\n        "answer": "UMANG provides access to a range of government services. Service availability varies by department.",\n        "url": "https://web.umang.gov.in/"\n    }\n]\n\nTRANSLATIONS = {\n    "Hindi": "यह SmartGov AI का डेमो उत्तर है। कृपया संबंधित सरकारी पोर्टल पर नवीनतम जानकारी और पात्रता की पुष्टि करें।",\n    "Marathi": "हे SmartGov AI चे प्रात्यक्षिक उत्तर आहे. कृपया संबंधित अधिकृत सरकारी पोर्टलवर नवीनतम माहिती आणि पात्रता तपासा."\n}\n\ndef answer_question(question, language="English"):\n    q = question.lower()\n    for item in FAQS:\n        if any(keyword in q for keyword in item["keywords"]):\n            answer = item["answer"]\n            if language in ("Hindi", "Marathi"):\n                answer = TRANSLATIONS[language] + "\\n\\n" + answer\n            return {\n                "answer": answer,\n                "source_url": item["url"],\n                "language": language,\n                "disclaimer": "Prototype FAQ matching; not an official government response."\n            }\n    fallback = "I couldn\'t find a matching topic in the demo FAQ. Try asking about scholarships, farmer support, application tracking, income certificates, grievances, or UMANG."\n    if language in ("Hindi", "Marathi"):\n        fallback = TRANSLATIONS[language] + "\\n\\n" + fallback\n    return {\n        "answer": fallback,\n        "source_url": "https://www.india.gov.in/",\n        "language": language,\n        "disclaimer": "Verify important information through official sources."\n    }\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('services/schemes.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('from services.storage import get_db\n\ndef search_schemes(query="", audience="citizen"):\n    q = query.lower().strip()\n    with get_db() as db:\n        rows = db.execute("SELECT * FROM schemes ORDER BY name").fetchall()\n    results = []\n    for row in rows:\n        item = dict(row)\n        searchable = " ".join([\n            item["name"], item["description"], item["audience"], item["keywords"]\n        ]).lower()\n        audience_match = audience in ("all", "any", "") or item["audience"] == audience or item["audience"] == "citizen"\n        query_match = not q or q in searchable or any(word in searchable for word in q.split())\n        if audience_match and query_match:\n            results.append({\n                "name": item["name"],\n                "description": item["description"],\n                "audience": item["audience"],\n                "official_url": item["official_url"],\n                "eligibility_note": item["eligibility_note"]\n            })\n    return results\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('services/tracker.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('from services.storage import get_db\n\ndef get_application(application_id):\n    with get_db() as db:\n        row = db.execute(\n            "SELECT application_id, applicant_name, service_name, status, last_updated, next_step "\n            "FROM applications WHERE application_id = ?",\n            (application_id.strip().upper(),)\n        ).fetchone()\n    return dict(row) if row else None\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('services/grievances.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('from datetime import datetime\nimport re\nfrom services.storage import get_db\n\nVALID_STATUSES = {"Submitted", "In Progress", "Resolved", "Closed"}\n\ndef classify_grievance(description):\n    text = description.lower()\n    rules = [\n        (["water", "leak", "pipeline", "drinking water"], "Water Supply", "Water Department"),\n        (["garbage", "waste", "cleanliness", "drain", "sewage"], "Sanitation", "Municipal Sanitation"),\n        (["road", "pothole", "streetlight", "street light"], "Roads and Infrastructure", "Public Works Department"),\n        (["electricity", "power cut", "electric"], "Electricity", "Electricity Department"),\n        (["pension", "benefit", "payment"], "Welfare Benefits", "Social Welfare Department"),\n    ]\n    for keywords, category, department in rules:\n        if any(k in text for k in keywords):\n            return category, department\n    return "General", "Public Grievance Office"\n\ndef create_grievance(citizen_name, description, locality=""):\n    category, department = classify_grievance(description)\n    urgent_terms = ["urgent", "emergency", "danger", "hazard", "life-threatening"]\n    priority = "High (review recommended)" if any(term in description.lower() for term in urgent_terms) else "Normal (review recommended)"\n    now = datetime.now().isoformat(timespec="seconds")\n    normalized = re.sub(r"\\W+", " ", description.lower()).strip()\n    duplicate = None\n    with get_db() as db:\n        recent = db.execute(\n            "SELECT ticket_id, description FROM grievances ORDER BY id DESC LIMIT 100"\n        ).fetchall()\n        for row in recent:\n            previous = re.sub(r"\\W+", " ", row["description"].lower()).strip()\n            if normalized and (normalized == previous or normalized in previous or previous in normalized):\n                duplicate = row["ticket_id"]\n                break\n        count = db.execute("SELECT COUNT(*) FROM grievances").fetchone()[0] + 1\n        ticket_id = f"GR-{2000 + count}"\n        db.execute("""\n            INSERT INTO grievances\n            (ticket_id, citizen_name, description, category, priority, department,\n             locality, status, possible_duplicate, created_at, updated_at)\n            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n        """, (ticket_id, citizen_name or "Demo Citizen", description, category, priority,\n              department, locality, "Submitted", duplicate, now, now))\n        db.commit()\n        row = db.execute("SELECT * FROM grievances WHERE ticket_id = ?", (ticket_id,)).fetchone()\n    return dict(row)\n\ndef list_grievances():\n    with get_db() as db:\n        rows = db.execute("SELECT * FROM grievances ORDER BY id DESC").fetchall()\n    return [dict(row) for row in rows]\n\ndef update_grievance(ticket_id, status):\n    if status not in VALID_STATUSES:\n        return None\n    now = datetime.now().isoformat(timespec="seconds")\n    with get_db() as db:\n        cursor = db.execute(\n            "UPDATE grievances SET status = ?, updated_at = ? WHERE ticket_id = ?",\n            (status, now, ticket_id)\n        )\n        db.commit()\n        if cursor.rowcount == 0:\n            return None\n        row = db.execute("SELECT * FROM grievances WHERE ticket_id = ?", (ticket_id,)).fetchone()\n    return dict(row)\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('services/dashboard.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('from services.storage import get_db\n\ndef dashboard_data():\n    with get_db() as db:\n        total_apps = db.execute("SELECT COUNT(*) FROM applications").fetchone()[0]\n        pending_apps = db.execute(\n            "SELECT COUNT(*) FROM applications WHERE status != \'Approved\'"\n        ).fetchone()[0]\n        total_grievances = db.execute("SELECT COUNT(*) FROM grievances").fetchone()[0]\n        high_priority = db.execute(\n            "SELECT COUNT(*) FROM grievances WHERE priority LIKE \'High%\'"\n        ).fetchone()[0]\n        app_rows = db.execute(\n            "SELECT status, COUNT(*) AS count FROM applications GROUP BY status"\n        ).fetchall()\n        grievance_rows = db.execute(\n            "SELECT category, COUNT(*) AS count FROM grievances GROUP BY category"\n        ).fetchall()\n        recent = db.execute(\n            "SELECT ticket_id, category, priority, department, status, created_at "\n            "FROM grievances ORDER BY id DESC LIMIT 10"\n        ).fetchall()\n    return {\n        "kpis": {\n            "total_applications": total_apps,\n            "pending_applications": pending_apps,\n            "total_grievances": total_grievances,\n            "high_priority_recommendations": high_priority\n        },\n        "application_statuses": [dict(r) for r in app_rows],\n        "grievance_categories": [dict(r) for r in grievance_rows],\n        "recent_grievances": [dict(r) for r in recent],\n        "disclaimer": "All figures are from this local prototype database, not live government systems."\n    }\n', encoding='utf-8')
print('Created', p)

In [ ]:
from pathlib import Path
p = Path('templates/index.html')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('<!doctype html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>SmartGov AI | Citizen Services</title>\n  <link rel="stylesheet" href="{{ url_for(\'static\', filename=\'style.css\') }}">\n</head>\n<body>\n<header class="topbar">\n  <a class="brand" href="#"><span class="brand-mark">S</span><span>SmartGov <b>AI</b><small>Citizen Services Portal</small></span></a>\n  <nav><a href="#assistant">Assistant</a><a href="#schemes">Schemes</a><a href="#tracker">Track</a><a href="#grievances">Grievances</a><a href="#dashboard">Dashboard</a></nav>\n</header>\n<main>\n  <section class="hero">\n    <div class="hero-copy">\n      <p class="eyebrow">SMARTER SERVICES · BETTER GOVERNANCE</p>\n      <h1>Government services,<br><span>made simpler.</span></h1>\n      <p>Find public-service information, explore schemes, track demo applications, and submit a grievance in one place.</p>\n      <a class="button primary" href="#services">Explore services</a>\n      <p class="disclaimer">Prototype for demonstration only. Not an official government portal.</p>\n    </div>\n    <div class="hero-panel">\n      <div class="panel-top"><span class="status-dot"></span> Service overview <span class="live-tag">DEMO DATA</span></div>\n      <div class="hero-stat"><strong>05</strong><span>connected service modules</span></div>\n      <div class="mini-row"><span>Citizen Assistant</span><b>Available</b></div>\n      <div class="mini-row"><span>Scheme Finder</span><b>Available</b></div>\n      <div class="mini-row"><span>Application Tracker</span><b>Sample records</b></div>\n      <div class="mini-row"><span>Grievance Management</span><b>Local tickets</b></div>\n    </div>\n  </section>\n\n  <section id="services" class="section">\n    <div class="section-heading"><div><p class="eyebrow">ONE PORTAL, FIVE TOOLS</p><h2>Citizen service centre</h2></div><p>Choose a module to get started.</p></div>\n    <div class="service-grid">\n      <a class="service-card" href="#assistant"><span class="service-icon">✦</span><h3>AI Citizen Assistant</h3><p>Find guidance and official resources.</p><span class="card-link">Ask a question →</span></a>\n      <a class="service-card" href="#schemes"><span class="service-icon">⌕</span><h3>Scheme Finder</h3><p>Search official scheme links across banking, insurance, housing, livelihoods and entrepreneurship.</p><span class="card-link">Find schemes →</span></a>\n      <a class="service-card" href="#tracker"><span class="service-icon">↗</span><h3>Application Tracker</h3><p>Check fictional reference records.</p><span class="card-link">Track application →</span></a>\n      <a class="service-card" href="#grievances"><span class="service-icon">≋</span><h3>Grievance Management</h3><p>Create and follow local demo tickets.</p><span class="card-link">Submit grievance →</span></a>\n      <a class="service-card" href="#dashboard"><span class="service-icon">▦</span><h3>Governance Dashboard</h3><p>View prototype data and ticket status.</p><span class="card-link">View dashboard →</span></a>\n    </div>\n  </section>\n\n  <section id="assistant" class="section split-section">\n    <div class="section-intro"><p class="eyebrow">01 / CITIZEN SUPPORT</p><h2>AI Citizen Assistant</h2><p>Ask about common public services. Responses use curated keyword-matched FAQs.</p></div>\n    <div class="module-card">\n      <label for="assistant-language">Response language</label>\n      <select id="assistant-language"><option>English</option><option>Hindi</option><option>Marathi</option></select>\n      <label for="assistant-question">Your question</label>\n      <textarea id="assistant-question" rows="3" placeholder="e.g. Where can I find scholarship information?"></textarea>\n      <button class="button primary" id="assistant-submit">Ask assistant</button>\n      <div id="assistant-result" class="result hidden" aria-live="polite"></div>\n    </div>\n  </section>\n\n  <section id="schemes" class="section split-section">\n    <div class="section-intro"><p class="eyebrow">02 / DISCOVER SUPPORT</p><h2>Personalized Scheme Finder</h2><p>Explore verified government scheme portals by keyword or beneficiary group. Links open the official site for current rules and applications.</p></div>\n    <div class="module-card">\n      <label for="scheme-audience">I am looking for</label>\n      <select id="scheme-audience"><option value="citizen">Any citizen</option><option value="student">Student</option><option value="farmer">Farmer</option><option value="worker">Unorganised worker</option><option value="business">Entrepreneur / vendor</option><option value="all">All audiences</option></select>\n      <label for="scheme-query">Keyword</label>\n      <input id="scheme-query" placeholder="PMJDY, housing, crop insurance, pension, startup...">\n      <button class="button primary" id="scheme-submit">Search schemes</button>\n      <div id="scheme-results" class="stack-results"></div>\n    </div>\n  </section>\n\n  <section id="tracker" class="section split-section">\n    <div class="section-intro"><p class="eyebrow">03 / APPLICATION STATUS</p><h2>Smart Application Tracker</h2><p>Use a sample reference such as <code>SG-1001</code>, <code>SG-1002</code>, or <code>SG-1003</code>.</p></div>\n    <div class="module-card">\n      <label for="application-id">Application reference number</label>\n      <div class="inline-form"><input id="application-id" placeholder="SG-1001"><button class="button primary" id="tracker-submit">Track</button></div>\n      <div id="tracker-result" class="result hidden" aria-live="polite"></div>\n    </div>\n  </section>\n\n  <section id="grievances" class="section split-section">\n    <div class="section-intro"><p class="eyebrow">04 / CITIZEN FEEDBACK</p><h2>Grievance Management</h2><p>Submit a demo complaint. The prototype categorizes it and creates a local ticket; it does not send it to a government office.</p></div>\n    <div class="module-card">\n      <label for="citizen-name">Name (optional)</label><input id="citizen-name" placeholder="Demo Citizen">\n      <label for="locality">Locality (optional)</label><input id="locality" placeholder="Area or locality">\n      <label for="grievance-description">Describe the issue</label><textarea id="grievance-description" rows="4" placeholder="e.g. Water is leaking from the pipeline near my street."></textarea>\n      <button class="button primary" id="grievance-submit">Submit demo grievance</button>\n      <div id="grievance-result" class="result hidden" aria-live="polite"></div>\n      <div class="list-heading"><h3>Recent tickets</h3><button class="button secondary small" id="refresh-grievances">Refresh</button></div>\n      <div id="grievance-list" class="stack-results"></div>\n    </div>\n  </section>\n\n  <section id="dashboard" class="section">\n    <div class="section-heading"><div><p class="eyebrow">05 / SERVICE INSIGHTS</p><h2>Smart Governance Dashboard</h2></div><button class="button secondary" id="refresh-dashboard">Refresh dashboard</button></div>\n    <p class="disclaimer">Statistics below are calculated from local demo records only.</p>\n    <div id="dashboard-kpis" class="kpi-grid"></div>\n    <div class="dashboard-grid">\n      <div class="module-card"><h3>Application statuses</h3><div id="application-status-chart" class="simple-chart"></div></div>\n      <div class="module-card"><h3>Grievance categories</h3><div id="grievance-category-chart" class="simple-chart"></div></div>\n    </div>\n    <div class="module-card table-card"><h3>Recent grievance tickets</h3><div class="table-wrap"><table><thead><tr><th>Ticket</th><th>Category</th><th>Priority</th><th>Status</th><th>Update status</th></tr></thead><tbody id="dashboard-tickets"></tbody></table></div></div>\n  </section>\n</main>\n<footer><strong>SmartGov AI</strong><span>Educational prototype · Verify services through official government sources.</span></footer>\n<script src="{{ url_for(\'static\', filename=\'app.js\') }}"></script>\n</body>\n</html>\n', encoding='utf-8')
print('Updated', p)

In [ ]:
from pathlib import Path
p = Path('static/style.css')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(':root{--navy:#102a43;--navy2:#183f5d;--teal:#087f8c;--teal-light:#e5f6f5;--ink:#243b53;--muted:#627d98;--line:#d9e2ec;--bg:#f6f9fc;--white:#fff;--shadow:0 14px 36px rgba(16,42,67,.08)}\n*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:90px}body{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif;line-height:1.6}.topbar{height:auto;min-height:76px;padding:14px max(5vw,20px);display:flex;align-items:center;justify-content:space-between;gap:24px;background:#fff;border-bottom:1px solid var(--line);position:sticky;top:0;z-index:10}.brand{display:flex;align-items:center;gap:11px;color:var(--navy);text-decoration:none;font-weight:750;font-size:18px;line-height:1.2}.brand b{color:var(--teal)}.brand small{display:block;font-size:11px;color:var(--muted);font-weight:500;letter-spacing:.04em;margin-top:4px}.brand-mark{width:42px;height:42px;display:grid;place-items:center;background:var(--navy);color:#fff;border-radius:13px;font-size:23px}.topbar nav{display:flex;gap:22px;flex-wrap:wrap}.topbar nav a{font-size:13px;color:var(--muted);text-decoration:none;font-weight:650}.topbar nav a:hover{color:var(--teal)}main{max-width:1180px;margin:auto;padding:0 26px}.hero{display:grid;grid-template-columns:1.15fr .85fr;gap:60px;align-items:center;padding:78px 0 70px}.eyebrow{font-size:11px;font-weight:800;letter-spacing:.14em;color:var(--teal);margin:0 0 12px}.hero h1{font-size:clamp(40px,5vw,61px);line-height:1.08;letter-spacing:-.045em;color:var(--navy);margin:0 0 20px}.hero h1 span{color:var(--teal)}.hero-copy>p:not(.eyebrow):not(.disclaimer){max-width:540px;color:var(--muted);font-size:17px;margin-bottom:28px}.button{border:0;border-radius:9px;padding:12px 17px;font-weight:750;font-size:13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;justify-content:center;transition:transform .15s,background .15s}.button:hover{transform:translateY(-1px)}.primary{background:var(--teal);color:white}.primary:hover{background:#066a75}.secondary{background:#edf4f8;color:var(--navy)}.small{padding:7px 11px;font-size:12px}.disclaimer{font-size:11px;color:var(--muted);margin-top:18px}.hero-panel,.module-card,.service-card{background:white;border:1px solid var(--line);border-radius:17px;box-shadow:var(--shadow)}.hero-panel{padding:26px}.panel-top{display:flex;align-items:center;gap:8px;font-size:13px;font-weight:750;color:var(--navy);padding-bottom:22px;border-bottom:1px solid var(--line)}.status-dot{width:8px;height:8px;border-radius:50%;background:#1a9c82}.live-tag{margin-left:auto;font-size:9px;letter-spacing:.08em;padding:4px 7px;border-radius:5px;background:#fff4dc;color:#936b12}.hero-stat{display:flex;align-items:baseline;gap:13px;padding:24px 0;color:var(--muted);font-size:13px}.hero-stat strong{font-size:46px;line-height:1;color:var(--navy)}.mini-row{display:flex;justify-content:space-between;gap:12px;border-top:1px solid #edf2f7;padding:13px 0;font-size:12px}.mini-row b{color:var(--teal);font-size:11px}.section{padding:45px 0 55px}.section-heading{display:flex;align-items:end;justify-content:space-between;gap:20px;margin-bottom:25px}.section h2{font-size:clamp(27px,3vw,36px);line-height:1.2;color:var(--navy);letter-spacing:-.025em;margin:0}.section-heading>p,.section-heading>div+ p{color:var(--muted);font-size:13px;margin:0}.service-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}.service-card{padding:23px;text-decoration:none;color:inherit;transition:transform .2s,border-color .2s}.service-card:hover{transform:translateY(-4px);border-color:#8acbca}.service-icon{display:grid;place-items:center;width:43px;height:43px;border-radius:12px;background:var(--teal-light);color:var(--teal);font-size:24px}.service-card h3{font-size:16px;color:var(--navy);margin:16px 0 6px}.service-card p{font-size:13px;color:var(--muted);margin:0 0 15px}.card-link{font-size:12px;font-weight:800;color:var(--teal)}.split-section{display:grid;grid-template-columns:.75fr 1.25fr;gap:55px;align-items:start;border-top:1px solid var(--line)}.section-intro{padding-top:10px}.section-intro>p:last-child{font-size:14px;color:var(--muted);max-width:340px}.module-card{padding:25px;min-width:0}.module-card h3{font-size:16px;color:var(--navy);margin:0 0 15px}label{display:block;font-size:12px;font-weight:750;color:var(--navy);margin:14px 0 7px}input,textarea,select{font:inherit;font-size:13px;color:var(--ink);background:white;border:1px solid #cbd7e3;border-radius:8px;padding:11px 12px;width:100%;outline:none}input:focus,textarea:focus,select:focus{border-color:var(--teal);box-shadow:0 0 0 3px rgba(8,127,140,.1)}textarea{resize:vertical;margin-bottom:12px}.module-card>.button{margin-top:7px}.result{margin-top:17px;background:#f1f8fa;border:1px solid #d5ebed;border-radius:10px;padding:15px;font-size:13px;overflow-wrap:anywhere}.result h3{margin:0 0 8px}.result p{margin:5px 0}.result a,.stack-results a{color:var(--teal);font-weight:700}.hidden{display:none!important}.inline-form{display:flex;gap:10px}.inline-form input{flex:1;min-width:0}.inline-form .button{white-space:nowrap}.stack-results{display:grid;gap:10px;margin-top:14px}.scheme-result,.ticket-item{border:1px solid var(--line);border-radius:10px;padding:14px;font-size:12px}.scheme-result h4,.ticket-item h4{margin:0 0 5px;font-size:14px;color:var(--navy)}.scheme-result p,.ticket-item p{margin:5px 0;color:var(--muted)}.list-heading{display:flex;justify-content:space-between;align-items:center;margin-top:24px}.list-heading h3{margin:0}.kpi-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:22px 0}.kpi{padding:20px;background:white;border:1px solid var(--line);border-radius:14px}.kpi span{display:block;color:var(--muted);font-size:12px}.kpi strong{display:block;color:var(--navy);font-size:31px;margin-top:5px}.dashboard-grid{display:grid;grid-template-columns:1fr 1fr;gap:16px}.simple-chart{display:grid;gap:14px}.bar-label{display:flex;justify-content:space-between;gap:10px;font-size:12px;margin-bottom:5px}.bar-track{height:8px;background:#e8eff4;border-radius:99px;overflow:hidden}.bar-fill{height:100%;background:var(--teal);border-radius:99px}.table-card{margin-top:16px}.table-wrap{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:12px}th,td{text-align:left;padding:12px 10px;border-bottom:1px solid var(--line);white-space:nowrap}th{color:var(--muted);font-weight:750}td select{min-width:125px;padding:7px;font-size:11px}td .button{padding:7px 9px;font-size:11px}footer{margin-top:45px;background:var(--navy);color:#fff;padding:25px max(5vw,20px);display:flex;justify-content:space-between;gap:15px;flex-wrap:wrap;font-size:12px}footer span{color:#c0d1df}\n@media(max-width:850px){.topbar{align-items:flex-start;flex-direction:column}.topbar nav{gap:14px}.hero{grid-template-columns:1fr;gap:28px;padding-top:50px}.service-grid{grid-template-columns:repeat(2,1fr)}.split-section{grid-template-columns:1fr;gap:20px}.kpi-grid{grid-template-columns:repeat(2,1fr)}}@media(max-width:520px){main{padding:0 16px}.topbar nav{gap:10px}.topbar nav a{font-size:11px}.service-grid,.dashboard-grid{grid-template-columns:1fr}.section-heading{align-items:flex-start;flex-direction:column}.hero h1{font-size:43px}.module-card{padding:18px}.inline-form{flex-direction:column}.kpi strong{font-size:26px}}\n\n/* Scheme directory refinements */\n.scheme-result{background:#fff;border:1px solid var(--line);border-radius:13px;padding:18px;margin:12px 0;box-shadow:0 6px 18px rgba(16,42,67,.04)}\n.scheme-result h4{font-size:16px;line-height:1.35;color:var(--navy);margin:0 0 8px}\n.scheme-result p{font-size:13px;color:var(--muted);margin:7px 0}\n.scheme-result a{display:inline-flex;margin-top:8px;color:var(--teal);font-weight:800;font-size:13px;text-decoration:none}\n.scheme-result a:hover{text-decoration:underline}\n#scheme-results:empty{display:none}\n#scheme-results:not(:empty){margin-top:16px}\n@media(max-width:760px){.topbar{align-items:flex-start;flex-direction:column}.topbar nav{gap:12px}.hero{grid-template-columns:1fr;gap:28px;padding-top:48px}.service-grid{grid-template-columns:1fr}.split-section{grid-template-columns:1fr}.section-heading{align-items:flex-start;flex-direction:column}main{padding:0 18px}}\n', encoding='utf-8')
print('Updated', p)

In [ ]:
from pathlib import Path
p = Path('static/app.js')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('const $ = (id) => document.getElementById(id);\n\nasync function api(url, options = {}) {\n  const response = await fetch(url, {\n    headers: { "Content-Type": "application/json", ...(options.headers || {}) },\n    ...options\n  });\n  const data = await response.json().catch(() => ({}));\n  if (!response.ok) throw new Error(data.error || `Request failed (${response.status})`);\n  return data;\n}\nfunction showMessage(target, message, isError = false) {\n  target.classList.remove("hidden");\n  target.innerHTML = "";\n  const p = document.createElement("p");\n  p.textContent = message;\n  if (isError) p.style.color = "#a12626";\n  target.appendChild(p);\n}\nfunction addText(parent, tag, text, className = "") {\n  const el = document.createElement(tag);\n  el.textContent = text ?? "";\n  if (className) el.className = className;\n  parent.appendChild(el);\n  return el;\n}\nfunction addLink(parent, label, url) {\n  const a = document.createElement("a");\n  a.textContent = label;\n  a.href = url;\n  a.target = "_blank";\n  a.rel = "noopener noreferrer";\n  parent.appendChild(a);\n}\nfunction renderResult(target, data) {\n  target.classList.remove("hidden");\n  target.innerHTML = "";\n  Object.entries(data).forEach(([key, value]) => {\n    if (value === null || value === undefined || value === "") return;\n    const label = key.replaceAll("_", " ").replace(/\\b\\w/g, c => c.toUpperCase());\n    addText(target, "p", `${label}: ${value}`);\n  });\n}\n$("assistant-submit").addEventListener("click", async () => {\n  const question = $("assistant-question").value.trim();\n  if (!question) return showMessage($("assistant-result"), "Please enter a question.", true);\n  const target = $("assistant-result");\n  showMessage(target, "Searching the demo FAQ...");\n  try {\n    const data = await api("/api/assistant", {\n      method: "POST",\n      body: JSON.stringify({ question, language: $("assistant-language").value })\n    });\n    target.innerHTML = "";\n    addText(target, "h3", "Assistant response");\n    addText(target, "p", data.answer);\n    addText(target, "p", data.disclaimer);\n    addLink(target, "Open official resource", data.source_url);\n  } catch (error) { showMessage(target, error.message, true); }\n});\n\n$("scheme-submit").addEventListener("click", async () => {\n  const target = $("scheme-results");\n  target.innerHTML = "Searching...";\n  try {\n    const data = await api("/api/schemes", {\n      method: "POST",\n      body: JSON.stringify({ query: $("scheme-query").value.trim(), audience: $("scheme-audience").value })\n    });\n    target.innerHTML = "";\n    if (!data.results.length) return showMessage(target, "No matching demo schemes found. Try another keyword or choose All audiences.");\n    data.results.forEach(item => {\n      const card = document.createElement("article"); card.className = "scheme-result";\n      addText(card, "h4", item.name);\n      addText(card, "p", item.description);\n      addText(card, "p", `Audience: ${item.audience}`);\n      addText(card, "p", item.eligibility_note);\n      addLink(card, "Visit official government portal ↗", item.official_url);\n      target.appendChild(card);\n    });\n  } catch (error) { showMessage(target, error.message, true); }\n});\n\n$("tracker-submit").addEventListener("click", async () => {\n  const id = $("application-id").value.trim();\n  const target = $("tracker-result");\n  if (!id) return showMessage(target, "Enter an application reference.", true);\n  showMessage(target, "Looking up demo record...");\n  try {\n    const data = await api(`/api/applications/${encodeURIComponent(id)}`);\n    renderResult(target, { application_id: data.application_id, service_name: data.service_name, status: data.status, last_updated: data.last_updated, next_step: data.next_step, notice: "Fictional demonstration record" });\n  } catch (error) { showMessage(target, error.message, true); }\n});\n\n$("grievance-submit").addEventListener("click", async () => {\n  const description = $("grievance-description").value.trim();\n  const target = $("grievance-result");\n  if (description.length < 8) return showMessage(target, "Please describe the issue in at least 8 characters.", true);\n  showMessage(target, "Creating local demo ticket...");\n  try {\n    const data = await api("/api/grievances", {\n      method: "POST",\n      body: JSON.stringify({\n        citizen_name: $("citizen-name").value.trim() || "Demo Citizen",\n        locality: $("locality").value.trim(),\n        description\n      })\n    });\n    renderResult(target, {\n      ticket_id: data.ticket_id, category: data.category, priority: data.priority,\n      department: data.department, status: data.status,\n      possible_duplicate: data.possible_duplicate || "No likely duplicate detected",\n      notice: "Saved locally only; not sent to a government department."\n    });\n    $("grievance-description").value = "";\n    await loadGrievances();\n    await loadDashboard();\n  } catch (error) { showMessage(target, error.message, true); }\n});\n\nasync function loadGrievances() {\n  const target = $("grievance-list");\n  target.innerHTML = "Loading tickets...";\n  try {\n    const data = await api("/api/grievances");\n    target.innerHTML = "";\n    if (!data.grievances.length) return showMessage(target, "No demo grievances have been submitted yet.");\n    data.grievances.slice(0, 8).forEach(item => {\n      const card = document.createElement("article"); card.className = "ticket-item";\n      addText(card, "h4", `${item.ticket_id} · ${item.category}`);\n      addText(card, "p", `${item.priority} — ${item.status}`);\n      addText(card, "p", item.description);\n      if (item.possible_duplicate) addText(card, "p", `Possible duplicate of ${item.possible_duplicate}`);\n      target.appendChild(card);\n    });\n  } catch (error) { showMessage(target, error.message, true); }\n}\n$("refresh-grievances").addEventListener("click", loadGrievances);\n\nfunction renderBars(target, rows, labelKey) {\n  target.innerHTML = "";\n  if (!rows.length) return addText(target, "p", "No data available yet.");\n  const max = Math.max(...rows.map(r => r.count), 1);\n  rows.forEach(row => {\n    const wrap = document.createElement("div");\n    const label = document.createElement("div"); label.className = "bar-label";\n    addText(label, "span", row[labelKey]); addText(label, "strong", String(row.count));\n    wrap.appendChild(label);\n    const track = document.createElement("div"); track.className = "bar-track";\n    const fill = document.createElement("div"); fill.className = "bar-fill";\n    fill.style.width = `${(row.count / max) * 100}%`;\n    track.appendChild(fill); wrap.appendChild(track); target.appendChild(wrap);\n  });\n}\nasync function loadDashboard() {\n  try {\n    const data = await api("/api/dashboard");\n    const kpis = $("dashboard-kpis"); kpis.innerHTML = "";\n    [\n      ["Total applications", data.kpis.total_applications],\n      ["Pending applications", data.kpis.pending_applications],\n      ["Grievances", data.kpis.total_grievances],\n      ["High-priority flags", data.kpis.high_priority_recommendations]\n    ].forEach(([label, value]) => {\n      const card = document.createElement("div"); card.className = "kpi";\n      addText(card, "span", label); addText(card, "strong", String(value)); kpis.appendChild(card);\n    });\n    renderBars($("application-status-chart"), data.application_statuses, "status");\n    renderBars($("grievance-category-chart"), data.grievance_categories, "category");\n    const tbody = $("dashboard-tickets"); tbody.innerHTML = "";\n    if (!data.recent_grievances.length) {\n      const tr = document.createElement("tr"); const td = document.createElement("td");\n      td.colSpan = 5; td.textContent = "No grievance tickets yet."; tr.appendChild(td); tbody.appendChild(tr);\n    }\n    data.recent_grievances.forEach(ticket => {\n      const tr = document.createElement("tr");\n      [ticket.ticket_id, ticket.category, ticket.priority, ticket.status].forEach(value => {\n        const td = document.createElement("td"); td.textContent = value; tr.appendChild(td);\n      });\n      const td = document.createElement("td");\n      const select = document.createElement("select");\n      ["Submitted", "In Progress", "Resolved", "Closed"].forEach(status => {\n        const opt = document.createElement("option"); opt.value = status; opt.textContent = status;\n        if (status === ticket.status) opt.selected = true;\n        select.appendChild(opt);\n      });\n      const btn = document.createElement("button"); btn.className = "button secondary small"; btn.textContent = "Save";\n      btn.addEventListener("click", async () => {\n        try {\n          await api(`/api/grievances/${encodeURIComponent(ticket.ticket_id)}`, {\n            method: "PATCH", body: JSON.stringify({ status: select.value })\n          });\n          await loadDashboard(); await loadGrievances();\n        } catch (error) { alert(error.message); }\n      });\n      td.append(select, btn); tr.appendChild(td); tbody.appendChild(tr);\n    });\n  } catch (error) {\n    showMessage($("dashboard-kpis"), error.message, true);\n  }\n}\n$("refresh-dashboard").addEventListener("click", loadDashboard);\n\nloadGrievances();\nloadDashboard();\n', encoding='utf-8')
print('Updated', p)

In [ ]:
from pathlib import Path
p = Path('README.md')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('# SmartGov AI — Government scheme directory and citizen-services demo\n\nA Flask + SQLite educational prototype with a citizen assistant, scheme finder, sample application tracker, local grievance tickets and a demo dashboard.\n\n## Government scheme links added\nThe scheme finder includes PMJDY, PMJJBY, PMSBY, APY, PM-SYM, PMAY, PM SVANidhi, PM Vishwakarma, PMFBY, MUDRA, Startup India, PM-KISAN, UMANG and the National Scholarship Portal. Search by scheme name, keyword or audience.\n\n## Run locally\n1. Install Python 3.10+.\n2. In this folder run: `python -m pip install -r requirements.txt`\n3. Initialize the database: `python -m services.storage`\n4. Start the site: `python app.py`\n5. Open `http://127.0.0.1:5000`.\n\n## Important\nThis is a demonstration, not an official government portal. Scheme links open official portals for information. Application tracking and grievance tickets are local fictional demo data; they are not connected to government databases and grievances are not submitted to government departments. Always verify eligibility, deadlines, benefits and application procedures on the official portal.\n', encoding='utf-8')
print('Updated', p)

## 3. Initialize and verify SQLite database

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
from services.storage import init_db, get_db
init_db()
with get_db() as db:
    for table in ("applications", "schemes", "grievances"):
        count = db.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {count} record(s)")

## 4. Start the website

This cell runs Flask in the notebook process. Colab normally does not expose localhost directly to your browser. Use the optional tunnel cell below if you want a clickable public preview.

In [ ]:
from threading import Thread
from werkzeug.serving import make_server

class FlaskServerThread(Thread):
    def __init__(self, flask_app):
        super().__init__(daemon=True)
        self.server = make_server("0.0.0.0", 5000, flask_app)
        self.context = flask_app.app_context()
        self.context.push()
    def run(self):
        self.server.serve_forever()
    def shutdown(self):
        self.server.shutdown()

from app import app
init_db()
server = FlaskServerThread(app)
server.start()
print("Flask server is running on port 5000.")

## 5. Optional: create a public preview URL with Cloudflare Tunnel

Only run this if you want to preview the website in your browser. The URL is public to anyone who has it while the tunnel is active. Do not enter real personal information or sensitive data in this demo.

In [ ]:
!pip -q install pycloudflared

In [ ]:
from pycloudflared import try_cloudflare
tunnel = try_cloudflare(port=5000)
print(tunnel)

## 6. API smoke tests

These requests verify the modules without needing to click through the website.

In [ ]:
from app import app
client = app.test_client()

checks = [
    ("POST", "/api/assistant", {"question": "Where can I find scholarship information?", "language": "English"}),
    ("POST", "/api/schemes", {"query": "scholarship", "audience": "student"}),
    ("GET", "/api/applications/SG-1001", None),
    ("GET", "/api/dashboard", None),
]
for method, url, payload in checks:
    response = client.open(url, method=method, json=payload)
    print(method, url, "->", response.status_code)
    print(response.get_json())